In [1]:
#Import the required Libraries
import os
import numpy as np
from openai import OpenAI
import faiss
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
import json
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
#Load open Ai key
load_dotenv(dotenv_path=r'C:\Users\User\Agentic AI\.env')
open_api_key = os.getenv("OPEN_API_KEY")
openai_client = OpenAI(api_key=open_api_key,
                       project="proj_AiRCDdpGwAmt6xebs4kjwD8f")

In [3]:
#Load the meeting transcript
def load_meeting_text(filename):
    with open(filename,'r',encoding="utf-8") as file:
        meeting_text = file.read()

    print("Data loaded successfully!")
    return meeting_text

In [4]:
#Call the function tomload the data
meeting_text = load_meeting_text('meeting_transcript.txt')

Data loaded successfully!


In [5]:
def add_line_numbers(text):

    lines = text.splitlines()

    numbered_lines = []

    for i, line in enumerate(lines, start=1):

        if line.strip():

            numbered_lines.append(
                f"[Line {i}] {line.strip()}"
            )

    return "\n".join(numbered_lines)


numbered_transcript = add_line_numbers(
    meeting_text
)

In [6]:
#Function to chunk the text usint NLTK punkt, here we are creating chucks of 500 characters
def chunk_by_sent(text, max_chars=500):
    sentences = nltk.sent_tokenize(text)
    chunks = []
    current = ""

    for sentence in sentences:
        if len(current) + len(sentence) < max_chars:
            current += " " + sentence
        else:
            chunks.append(current.strip())
            current = sentence
    if current:
        chunks.append(current.strip())
    print("Number of chunks:", len(chunks))
    return chunks

In [7]:
#Chunk text nefore proceeding to Embedding in order to give better context to RAG
chunks = chunk_by_sent(numbered_transcript)
print(chunks[:3])

Number of chunks: 8
["[Line 1] MEETING TRANSCRIPT — PRODUCT DEVELOPMENT TEAM\n[Line 3] Sarah: Good morning everyone. Let's start our project status meeting. [Line 5] Sarah: Today I want us to review the remaining development work, discuss the project proposal, and identify the next action items. [Line 7] David: The database work is progressing well. [Line 9] David: I have completed the progress-tracking tables. [Line 11] David: I still need to finish the remaining database changes. [Line 13] Sarah: Great.", "Please make sure the database changes are tracked as an action item. [Line 15] David: Yes, I'll take care of those database changes. [Line 17] Priya: I've also been testing the content-generation and question-generation prompts. [Line 19] Priya: The results are improving, but I need to run a few more tests. [Line 21] Priya: I'll continue testing the prompts. [Line 23] Sarah: Good. Please include that as another action item. [Line 25] Michael: The initial UI is partially complete.",

In [8]:
#Function to create OpenAI embeddings
def create_embeddings(chunks):
    response = openai_client.embeddings.create(
        model = 'text-embedding-3-small',
        input =  chunks   
    )

    embeddings = [item.embedding for item in response.data]

    return np.array(embeddings).astype("float32")
  

In [9]:
#Call the embeddings function to  create the embeddings
embeddings = create_embeddings(chunks)

In [10]:
#Create FAISS vector store

# dimension of embedding
d = embeddings.shape[1]
# create index
index = faiss.IndexFlatL2(d)

index.add(embeddings)
print(f'Vectors stoored into FIASS index successfull!. Total vectors={index.ntotal}')

Vectors stoored into FIASS index successfull!. Total vectors=8


In [11]:
def context_retriever(text, k=3):
    response = openai_client.embeddings.create(
      model = "text-embedding-3-small",
      input = text
    )
    question_embeddings = np.array([response.data[0].embedding]).astype("float32")
    
    distance, indices = index.search(question_embeddings,k )

    retrieved_chunks = []

    for i in indices[0]:
        retrieved_chunks.append(chunks[i])

    return retrieved_chunks
        

In [12]:
# Define the LLM
llm = LLM(
    model="gpt-5-mini",
    api_key=open_api_key
)

As this involves humans review am creating 2 Crews.

Crew-1:
Meeting Agent,
Decision Agent,
Action Agent,

Crew-2:
Review Agent,
Summary Agent

In [13]:
#CREW-1
#Create the Agents
#Agent_1 = meeting_agent
meeting_agent = Agent(
    role = "Meeting Agent",
    goal = "Analyze the meeting notes/information and identify the main points of discussion",
    backstory = """You are an experienced meeting analyst. Who carefully examine the meeting information
    and identify the important topics and discussions.Please use only the information from the context""",
    llm = llm,
    verbose = True
)

#Agent_2 = Decision Agent
decision_agent = Agent(
    role = "Decision Extractor",
    goal = "Identify the decisions made during the meeting",
    backstory = """You are an expert in extracting confirmed decisions from the business meetings.
    Distinguish between discussions, suggestions, decisions. Please don't create new information""",
    llm = llm,
    verbose = True
)

#Agent_3 = Action Agent
action_agent = Agent(
    role = "Action Item Manager",
    goal = "Identify the action items, responsible people and deadlines in the transcipt",
    backstory = """You an expertise in Project Coordination.
    Please Extarct the following:
    task
    owner
    deadline
    evidence
    transcipt line reference

    If ower is not specified say 'owner - UnResolved'
    
    If a deadline is missing please say 'Deadline is not specified'. 
    Never give random deadline or never guess a deadline.

    Always return valid JSON
    """,
    llm = llm,
    verbose = True
)



In [14]:
#Create tasks for the agents
#1. Task for analysis

analysis_task = Task(

    description = """ Analyse the retrieved meeting information. 
    Meeting Transcript:{context}
   

    Produce concise meeting summary.

        Identify:
    - Main topics discussed
    - Important discussion points
    - Key outcome
    
    Support inportant points with transcript evidence & line numbers references.

    Please use only the information from the meeting context. Strictly avoid information outside 
    the retrieved context.
    """,

    expected_output = """
    A concise meeting summary with supporting
    evidence and transcript line references.
    """,
    agent = meeting_agent
    )

#2. Task for decision

decision_task = Task(
    description=""" Analyze the meeting transcript.

    Meeting Transcript:
    {context}

    Identify all the decisions made during the meeting

    For every decision provide the following:
        Discussion point.
        Eidence.
        Transcipt Line Reference.

    If no relevant information if found, then say: 'No relevant decisions found!'
    
    Only report the decisions supported by the context.
    """,    

    expected_output=""" A list of relevant meeting decisions with supporting evidences and transcipt line references.
      Say so, if there are no relevant decisions""",
    agent = decision_agent,

    context=[
        analysis_task
    ]
)

#3. Task for Action

action_task = Task(
    description = """Identify ALL specific action items from the meeting.
    
    Meeting Transcript: {context}
    
    Use the output from meeting_agent & decision agent to identify the action items
    
       
    For each action item you need to identify:

    Owner
    Task
    Deadline
    Evidence
    Line reference
    Clarification status

    If a deadline is not present, write:
    'Deadline not specified'.

    If a deadline is not present, write:
    'oNWER': 'UNRESOLVED'.

    CLARIFICATION STATUS:

    If owner AND deadline are known:

        clarification_status = "Not Required"

    If either owner OR deadline is missing:

        clarification_status = "Required"
    
    IMPORTANT: Never infer an owner from someone's role. Never infer a deadline

    Return only valid JSON.

    Use this structure:

    {{
        "action_items": [
            {{
                "task": "...",
                "owner": "...",
                "deadline": "...",
                "evidence": "...",
                "line_reference": "..."
                @clarification_status": "..."
            }}
        ]
    }}

    """,

    expected_output=""" Valid JSON containing the extracted action items""",
    agent =  action_agent,
    context=[
        analysis_task,
        decision_task
    ]
)


In [15]:
#Create crew_1
crew_1 = Crew(
    agents = [
        meeting_agent,
        decision_agent,
        action_agent
    ],
    tasks = [
        analysis_task,
        decision_task,
        action_task
    ],
    process = Process.sequential,
    verbose = True
)

In [16]:
#CREW-2
#Create Review & Summary Agent

#Agent_4 = Review Agent
review_agent = Agent(
    role = "Meeting Report Reviewer",
    goal = """Review all extracted meeting information and correct
    unsupported or inconsistent information before the
    final report is produced""",
    backstory = """ You are an experienced meeting reviewer.
    You're job is to verify the following:
    - Meeting summary
    - Decisions
    - Action items
    - Owners
    - Deadlines
    - Evidence
    - Transcript references
    - User clarifications
     
     If the user supplied a clarification, preserve it.

    If the user could not clarify something, preserve
    UNRESOLVED""",

    llm = llm,
    verbose = True
)

#Agent_5 = Summary Agent
summary_agent = Agent(
    role = "Meeting Summary Specialist",
    goal = """Generate the final MeetMind AI meeting report.""",
    backstory = """ You are an intelligent meeting assistant.

    Create a concise, structured final report.

    The report must contain:

    1. Meeting Summary
    2. Decisions
    3. Action Item Table
    4. Clarification Status
    5. Agent Activity Log

    Include evidence and transcript line references
    for decisions and action items.
    """,
    llm = llm,
    verbose = True
)

In [17]:
#Crew -2 tasks:
#4. Task for reviewing 
review_task = Task(
    description= """ Review the extracted meeting information.
    
      Meeting Transcipt: {context}

    Meeting Analysis: {analysis}
    Decisions: {decisions}
    FINAL ACTION ITEMS AFTER HUMAN CLARIFICATION:: {action_items}
    User Clarifications: {clarifications}

    Review the information carefully.

    Verify that:

    1. The summary is supported by the transcript.
    2. Decisions are explicitly supported.
    3. Action items are real tasks.
    4. Owners are not invented.
    5. Deadlines are not invented.
    6. User clarifications are incorporated.
    7. UNRESOLVED values remain UNRESOLVED.
    8. Decisions have transcript references.
    9. Action items have transcript references.
    10. Evidence is consistent with the transcript.

    IMPORTANT:

    The FINAL ACTION ITEMS contain the latest owner
    and deadline information.

    Treat those values as authoritative.

    Do not change a user-provided owner or deadline
    back to UNRESOLVED.

    Do not invent missing information.

    Correct any unsupported information before
    passing the result to the Summary Agent

     Correct any supported information. Never guess/invent information.
    """,

    expected_output= """
        Reviewed and corrected meeting report containing:
         -  Verified Summary
         - Discussion points
         - Decision points
         - Action items
         - Evidence
         - Transcript refernce
         - User clarifications
         - Unresolved information
        """,

        agent= review_agent
)

summarization_task = Task(
    description="""Create the final MeetMind AI meeting report.Use the Review Agent's output as the authoritative
    reviewed informatio

    Meeting Summary :{analysis}
    Decisions : {decisions}
    Action Items: {action_items}
    User Clarifications: {clarifications}
    
    Produce the following selections:
    1. Meeting summary: A concise meeting summary.
    2. Decisions: List Explicit decisions and each decision should include:
            - Decision
        - Evidences
        - Transcript References
    3. Action Items: Create a table containing:
        | Task | Owner | Deadline | Clarification Status |
    4. Evidence: For each action item include:
        -  Evidence
        - Transcript refrence
    5. Agent Activity Log: Show which agent performed each task.
        Example:
        Meeting Agent:
        Generated meeting summary

    IMPORTANT:

    If the user could not clarify an owner or deadline,
    display: UNRESOLVED.

    Never invent the information!

    """,
    expected_output= """A complete and concise meeting report containing:

    1. Meeting Summary
    2. Decisions with evidence and transcript references
    3. Action-item table
    4. Clarification status
    5. Evidence and transcript references
    6. Agent activity log
        """,
    agent = summary_agent,
    context = [review_task]

)

In [18]:
#Now lets create CREW-2
crew_2 = Crew(
    agents = [review_agent,
              summary_agent],
    tasks = [review_task,
             summarization_task],
    process = Process.sequential,
    verbose = True
)

In [19]:
#Function to extract JSON
def parse_action_items(action_output):
    text = str(action_output).strip()
    if text.startswith("'''json"):
        text = text[len("'''json")].strip()
    elif text.startswith("```"):
        text = text[len("```"):].strip()

    if text.endswith("```"):
        text = text[:-3].strip()
    try:
        data = json.loads(text)
        return data.get("action_items",[])
    except json.JSONDecodeError as e:
        print("WARNING: Action Agent didn't return VALID JSON!")
        print("JSON error:", e)
        print("Raw Action Agent Output:")
        print(text)

    return []
    

In [20]:
#Function to ask user for clarifications
def clarify_items(action_items):
    clarifications = []
    for item in action_items:
        task = item.get("task", "Unknown task")

        owner = item.get("owner", "UNRESOLVED")
        if ( not owner or owner.upper()=="UNRESOLVED"):
            answer = input(f"\n MeetMindAI needs some clarifications.\n"
                           f"Who is responsible for:\n"
                           f"{task}\n"
                           f"Enter the name or type 'unresolved':").strip()
            if answer.lower() == "unresolved":
                item["owner"] = "UNRESOLVED"
            else:
                item["ownner"] = answer

            clarifications.append(f"Owner clarification for {task}: {answer}")

        deadline = item.get("deadline","UNRESOLVED")
        if (not deadline or deadline.upper()=="UNRESOLVED"):
            answer = input(f"\n MeetMindAI needs some clarifications.\n"
                           f"What is the deadline for:\n"
                           f"{task}\n"
                           f"Enter a date or type 'unresolved':").strip()
            if answer.lower() == "unresolved":
                item["deadline"] = "UNRESOLVED"
            else:
                item["deadline"] = answer

        
            clarifications.append(f"Deadline clarification for {task}: {answer}")

        if (
            item["owner"].upper() == "UNRESOLVED"
            or
            item["deadline"].upper() == "UNRESOLVED"
        ):

            item["clarification_status"] = "UNRESOLVED"

        else:

            item["clarification_status"] = "RESOLVED"

    return(action_items, clarifications)

In [21]:
#Function to get the task outputs from the crews starting with crew-1
def get_crew1_outputs(result):
    analysis_output = ""
    decision_output = ""
    action_output = ""

    try:
        outputs = result.tasks_output
        for i, output in enumerate(outputs):
            print(f"\n--- Task {i + 1} ---")
            print("Output type:", type(output))
            print("Agent:", getattr(output, "agent", "Unknown"))
            print("Raw output:")
            print(getattr(output, "raw", ""))

        print("No of output tasks:",len(outputs))

        if len(outputs) >= 1:
            analysis_output = str(outputs[0].raw)

        if len(outputs) >= 2:
            decision_output = str(outputs[1].raw)

        if len(outputs) >= 3:
            action_output = str(outputs[2].raw)
    except Exception as e:
        print("ERROR: Could not read task outputs!",e)

    return(analysis_output, decision_output, action_output)

In [22]:
async def run_meetmind():
    activity_log =[]
    print("\n"+ "=" * 60)
    print("MeetMind AI - Multi Agent AI Assistant")
    print("\n"+ "=" * 60)

    print("\n\n Retrieving meeting information.....")

    #Retrive the data using RAG
    retrieved_chunks = context_retriever("meeting summary, decisions, action items and owners deadlines")
    context = "\n\n".join(retrieved_chunks)

    print("\n\n RAG retrieval completed")
    print("\n"+ "=" * 60)
    print("\n Crew - 1 Started....")
    print("\n"+ "=" * 60)

    activity_log.append("Meeting Agent Started...")
    activity_log.append("Decision Agent Started..")
    activity_log.append("Action Item Manager Started..")

    extraction_result = await crew_1.kickoff_async(
        inputs ={
            "context" : context
        }
    )

    (analysis_output, decision_output, action_output) = get_crew1_outputs(extraction_result)
    activity_log.append("Meeting Agent Generated Summar!")
    activity_log.append("Decision Agent Extracted/Identified decisions!")
    activity_log.append("Action Agent Ectracted Action Items!")

    action_items = parse_action_items(action_output)
    print(f"\n Lenght of extracted Action Items: {len(action_items)}")

    #Human Clarification
    action_items, clarifications =(
        clarify_items(action_items)
    )

    if clarifications:
        activity_log.append("Human In Loop: Collected the clarifications for the missing owner/deadlines")
    else:
        activity_log.append("Human In Loop: no clairifications required!")


    action_items_json = json.dumps(
        {
            "action_items" : action_items
        },
        indent = 2
    )

    clarification_json = json.dumps(
        clarifications,
        indent = 2
    )

    #CREW-2 started
    print("\n"+ "=" * 60)
    print("\n Crew - 2 - Review, Correct & Summariza Started....")
    print("\n"+ "=" * 60)

    review_result = await crew_2.kickoff_async(
        inputs = {
            "context" : context,
            "analysis" : analysis_output,
            "decisions" : decision_output,
            "action_items" : action_output,
            "clarifications" : clarification_json,

        }
    )

    activity_log.append("Review Agent Review and Correction Completed!")
    activity_log.append("Summary Agent Generated the Final Meeting Report Successfully!")

    from IPython.display import display, Markdown

    final_report = review_result.tasks_output[-1].raw
    print("\n"+"="* 70)
    print("Final MeetMind AI Report:")
    display(Markdown(final_report))
    final_report = review_result.tasks_output[1].raw    
    print("\n"+"="* 70)
    print("\n Please find the activity log as follows:")
    print("\n"+"="* 70)
    for activity in activity_log:
        print(activity)

In [23]:

#Start the MeetMindAI
await run_meetmind()



MeetMind AI - Multi Agent AI Assistant



 Retrieving meeting information.....


 RAG retrieval completed


 Crew - 1 Started....



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.15                                                                                       │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1fb4a450-960e-49a7-9125-1b925eef29dc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:  Analyse the retrieved meeting information.                                                              │
│      Meeting Transcript:[Line 65] Sarah: There is another task we need to capture. [Line 67] Sarah: Someone     │
│  needs to estimate the remaining development work. [Line 69] David: We haven't assigned that task to anyone     │
│  either. [Line 71] Priya: And we haven't set a deadline for the estimate. [Line 73] Michael: We can discuss     │
│  the assignment after this meeting. [Line 75] Sarah: Okay. Record both the owner and deadline as unresolved     │
│  for that task. [Line 77] Sarah: Let's summarize the action items.                                              │
│                                                                                                                 │
│  [Line 93] Sarah: The deadline for the project proposal also needs to be determined. [Line 95] Sarah: The       │
│  remaining development work also needs to be estimated. [Line 97] Sarah: The person responsible for the         │
│  estimate has not been assigned. [Line 99] Sarah: The deadline for completing the estimate has also not been    │
│  assigned. [Line 101] Sarah: These unresolved items should be clarified before we finalize the project plan.    │
│  [Line 103] Sarah: That's all for today's meeting. Thank you everyone.                                          │
│                                                                                                                 │
│  [Line 53] Michael: We should decide that before the project review. [Line 55] Sarah: Agreed. Let's leave the   │
│  owner unresolved for now and come back to it after we estimate the remaining work. [Line 57] Sarah: The        │
│  proposal also needs to be completed before the next project review. [Line 59] Sarah: We haven't decided the    │
│  exact date for the review yet. [Line 61] David: So the deadline for the proposal is not known yet. [Line 63]   │
│  Sarah: Correct. We'll determine the date later.                                                                │
│                                                                                                                 │
│                                                                                                                 │
│      Produce concise meeting summary.                                                                           │
│                                                                                                                 │
│          Identify:                                                                                              │
│      - Main topics discussed                                                                                    │
│      - Important discussion points                                                                              │
│      - Key outcome                                                                                              │
│                                                                                                                 │
│      Support inportant points with transcript evidence & line numbers references.                               │
│                                                                                                                 │
│      Please use only the information from the meeting context. Strictly avoid information outside               │
│      the retrieved context.                                                                                     │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Meeting Agent                                                                                           │
│                                                                                                                 │
│  Task:  Analyse the retrieved meeting information.                                                              │
│      Meeting Transcript:[Line 65] Sarah: There is another task we need to capture. [Line 67] Sarah: Someone     │
│  needs to estimate the remaining development work. [Line 69] David: We haven't assigned that task to anyone     │
│  either. [Line 71] Priya: And we haven't set a deadline for the estimate. [Line 73] Michael: We can discuss     │
│  the assignment after this meeting. [Line 75] Sarah: Okay. Record both the owner and deadline as unresolved     │
│  for that task. [Line 77] Sarah: Let's summarize the action items.                                              │
│                                                                                                                 │
│  [Line 93] Sarah: The deadline for the project proposal also needs to be determined. [Line 95] Sarah: The       │
│  remaining development work also needs to be estimated. [Line 97] Sarah: The person responsible for the         │
│  estimate has not been assigned. [Line 99] Sarah: The deadline for completing the estimate has also not been    │
│  assigned. [Line 101] Sarah: These unresolved items should be clarified before we finalize the project plan.    │
│  [Line 103] Sarah: That's all for today's meeting. Thank you everyone.                                          │
│                                                                                                                 │
│  [Line 53] Michael: We should decide that before the project review. [Line 55] Sarah: Agreed. Let's leave the   │
│  owner unresolved for now and come back to it after we estimate the remaining work. [Line 57] Sarah: The        │
│  proposal also needs to be completed before the next project review. [Line 59] Sarah: We haven't decided the    │
│  exact date for the review yet. [Line 61] David: So the deadline for the proposal is not known yet. [Line 63]   │
│  Sarah: Correct. We'll determine the date later.                                                                │
│                                                                                                                 │
│                                                                                                                 │
│      Produce concise meeting summary.                                                                           │
│                                                                                                                 │
│          Identify:                                                                                              │
│      - Main topics discussed                                                                                    │
│      - Important discussion points                                                                              │
│      - Key outcome                                                                                              │
│                                                                                                                 │
│      Support inportant points with transcript evidence & line numbers references.                               │
│                                                                                                                 │
│      Please use only the information from the meeting context. Strictly avoid information outside               │
│      the retrieved context.                            

...


Final MeetMind AI Report:


Final MeetMind AI Meeting Report

1) Meeting Summary
- Concise summary: The team discussed the project proposal deadline relative to the next project review (the review date — and therefore the proposal deadline — remains unset) and identified a task to estimate remaining development work which currently has no owner or deadline. The assignment of who will perform that estimate will be discussed after the meeting; meanwhile the owner and deadline for that estimate were recorded as UNRESOLVED. Sarah summarized action items and emphasized that all unresolved items must be clarified before finalizing the project plan. (Transcript evidence: Lines 53–63, 65–75, 93–101, 103.)

2) Decisions (each decision followed by evidence and transcript references)

1. Decision: The project proposal must be completed before the next project review.
   - Evidence: Sarah: "The proposal also needs to be completed before the next project review."
   - Transcript References: Lines 53, 55, 57

2. Decision: Leave the owner of the proposal/task unresolved for now and revisit assignment after estimating the remaining work.
   - Evidence: Sarah: "Agreed. Let's leave the owner unresolved for now and come back to it after we estimate the remaining work."
   - Transcript Reference: Line 55

3. Decision: The assignment of who will estimate the remaining development work will be discussed after the meeting.
   - Evidence: Michael: "We can discuss the assignment after this meeting."
   - Transcript Reference: Line 73

4. Decision: Record both the owner and the deadline as unresolved for the estimate task.
   - Evidence: Sarah: "Okay. Record both the owner and deadline as unresolved for that task."
   - Transcript Reference: Line 75

5. Decision: The unresolved items (project review date/proposal deadline; person to estimate remaining development work; deadline for that estimate) must be clarified before finalizing the project plan.
   - Evidence: Sarah: "These unresolved items should be clarified before we finalize the project plan."
   - Transcript References: Lines 93–101

6. Decision: The exact date for the project review (and therefore the proposal deadline) will be determined later.
   - Evidence: David: "So the deadline for the proposal is not known yet." Sarah: "Correct. We'll determine the date later."
   - Transcript References: Lines 61–63

3) Action Items (table)

| Task                                                                 | Owner   | Deadline  | Clarification Status                     |
|----------------------------------------------------------------------|--------:|----------:|------------------------------------------|
| Complete project proposal before the next project review             | Sarah   | UNRESOLVED| User-clarified owner (not in transcript) |
| Estimate the remaining development work                              | UNRESOLVED | UNRESOLVED| Required (no owner/deadline in transcript)|
| Assign person responsible for estimating remaining development work (discussion after meeting) | Priya   | UNRESOLVED| User-clarified owner (not in transcript) |
| Record owner and deadline as unresolved for the estimate task        | Michael | UNRESOLVED| User-clarified owner (not in transcript) |
| Clarify all unresolved items before finalizing the project plan     | UNRESOLVED | UNRESOLVED| Required (no owner/deadline in transcript)|

4) Evidence for each Action Item (evidence + transcript reference)

1. Task: Complete project proposal before the next project review
   - Evidence (transcript excerpts):
     - "The proposal also needs to be completed before the next project review." — Sarah
     - "We haven't decided the exact date for the review yet." — Sarah
     - "So the deadline for the proposal is not known yet." — David
     - "Correct. We'll determine the date later." — Sarah
   - Transcript References: Lines 53, 55, 57, 59, 61, 63, 93
   - Note: Owner "Sarah" is a user clarification (preserved). Deadline remains UNRESOLVED.

2. Task: Estimate the remaining development work
   - Evidence (transcript excerpts):
     - "There is another task we need to capture." — Sarah
     - "Someone needs to estimate the remaining development work." — Sarah
     - "We haven't assigned that task to anyone either." — David
     - "And we haven't set a deadline for the estimate." — Priya
   - Transcript References: Lines 65, 67, 69, 71, 95, 97
   - Note: Owner and deadline are UNRESOLVED per transcript.

3. Task: Assign the person responsible for estimating the remaining development work (discussion to occur after the meeting)
   - Evidence (transcript excerpts):
     - "We haven't assigned that task to anyone either." — David
     - "We can discuss the assignment after this meeting." — Michael
     - "The person responsible for the estimate has not been assigned." — Sarah
   - Transcript References: Lines 69, 73, 97
   - Note: Owner "Priya" is a user clarification (preserved). Deadline remains UNRESOLVED.

4. Task: Record owner and deadline as unresolved for the estimate task
   - Evidence (transcript excerpt):
     - "Okay. Record both the owner and deadline as unresolved for that task." — Sarah
   - Transcript Reference: Line 75
   - Note: Owner "Michael" is a user clarification (preserved). Deadline remains UNRESOLVED.

5. Task: Clarify all unresolved items (project review date/proposal deadline; assignment and deadline for estimate) before finalizing the project plan
   - Evidence (transcript excerpts):
     - "The deadline for the project proposal also needs to be determined." — Sarah
     - "The remaining development work also needs to be estimated." — Sarah
     - "The person responsible for the estimate has not been assigned." — Sarah
     - "These unresolved items should be clarified before we finalize the project plan." — Sarah
   - Transcript References: Lines 93, 95, 97, 99, 101
   - Note: Owner and deadline are UNRESOLVED per transcript.

5) Clarification Status (summary)
- Items currently UNRESOLVED (per transcript and preserved):
  - Project review date → UNRESOLVED (affects proposal deadline). Transcript: Lines 59, 61–63.
  - Proposal deadline → UNRESOLVED. Transcript: Lines 59, 61–63.
  - Owner for estimating remaining development work → UNRESOLVED in transcript (user left it UNRESOLVED; user later provided "UNRESOLVED" in User Clarifications).
  - Deadline for the estimate → UNRESOLVED. Transcript: Line 71.
  - Owner responsible for clarifying all unresolved items before finalizing project plan → UNRESOLVED.

- User-provided clarifications preserved (authoritative per instructions):
  - Owner for "Complete project proposal before the next project review": Sarah (User clarification; not in transcript).
  - Owner for "Assign person responsible for estimating the remaining development work": Priya (User clarification; not in transcript).
  - Owner for "Record owner and deadline as unresolved for the estimate task": Michael (User clarification; not in transcript).
  - Other owners/deadlines remain UNRESOLVED.

6) Agent Activity Log (who performed each task)

- Meeting Agent:
  - Conducted meeting and produced raw transcript (referenced lines: 53–103).
  - Generated initial notes captured in transcript.

- Review Agent:
  - Reviewed and corrected the meeting report; produced the Verified Summary, Decisions, and Final Action Items draft (authoritative reviewed information used here).
  - Identified transcript-supported evidence and compiled the decision/action-item list referenced above.

- User:
  - Provided additional owner clarifications for certain action items (preserved exactly and treated as authoritative where supplied):
    - Sarah as owner for proposal task
    - Priya as owner for assignment-of-estimator task
    - Michael as owner for recording owner/deadline as unresolved
    - Other clarifications marked UNRESOLVED as provided

- Meeting Summary Specialist (this assistant):
  - Generated this final MeetMind AI meeting report using the Review Agent’s output as authoritative and preserving user clarifications.

———

Notes / Next steps (concise)
- Required follow-up before project plan finalization:
  1. Determine project review date → sets proposal deadline (UNRESOLVED).
  2. Assign who will produce the estimate for remaining development work (user indicated Priya will be involved in assignment; transcript left it unresolved).
  3. Set deadline for the remaining-work estimate.
  4. Michael (per user clarification) should record the owner/deadline as unresolved in the tracking system until assignment/deadline are clarified.
- If you want the action items exported as machine-readable JSON for import into a tracker, I can provide that (preserving UNRESOLVED fields).



 Please find the activity log as follows:

Meeting Agent Started...
Decision Agent Started..
Action Item Manager Started..
Meeting Agent Generated Summar!
Decision Agent Extracted/Identified decisions!
Action Agent Ectracted Action Items!
Human In Loop: Collected the clarifications for the missing owner/deadlines
Review Agent Review and Correction Completed!
Summary Agent Generated the Final Meeting Report Successfully!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 14022bbc-4c60-41fc-80d1-8e5970129136                                                                       │
│  Final Output: Final MeetMind AI Meeting Report                                                                 │
│                                                                                                                 │
│  1) Meeting Summary                                                                                             │
│  - Concise summary: The team discussed the project proposal deadline relative to the next project review (the   │
│  review date — and therefore the proposal deadline — remains unset) and identified a task to estimate           │
│  remaining development work which currently has no owner or deadline. The assignment of who will perform that   │
│  estimate will be discussed after the meeting; meanwhile the owner and deadline for that estimate were          │
│  recorded as UNRESOLVED. Sarah summarized action items and emphasized that all unresolved items must be         │
│  clarified before finalizing the project plan. (Transcript evidence: Lines 53–63, 65–75, 93–101, 103.)          │
│                                                                                                                 │
│  2) Decisions (each decision followed by evidence and transcript references)                                    │
│                                                                                                                 │
│  1. Decision: The project proposal must be completed before the next project review.                            │
│     - Evidence: Sarah: "The proposal also needs to be completed before the next project review."                │
│     - Transcript References: Lines 53, 55, 57                                                                   │
│                                                                                                                 │
│  2. Decision: Leave the owner of the proposal/task unresolved for now and revisit assignment after estimating   │
│  the remaining work.                                                                                            │
│     - Evidence: Sarah: "Agreed. Let's leave the owner unresolved for now and come back to it after we estimate  │
│  the remaining work."                                                                                           │
│     - Transcript Reference: Line 55                                                                             │
│                                                                                                                 │
│  3. Decision: The assignment of who will estimate the remaining development work will be discussed after the    │
│  meeting.                                                                                                       │
│     - Evidence: Michael: "We can discuss the assignment after this meeting."                                    │
│     - Transcript Reference: Line 73                                                                             │
│                                                                                                                 │
│  4. Decision: Record both the owner and the deadline as unresolved for the estimate task.                       │
│     - Evidence: Sarah: "Okay. Record both the owner and deadline as unresolved for that task."                  │
│     - Transcript Reference: Line 75                   

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯